# 01. Подключения и ETL PostgreSQL → MinIO

Проверяет подключение к PostgreSQL и MinIO, затем запускает выгрузку данных в S3-совместимое хранилище с partitioning.

In [ ]:
import os
from sqlalchemy import create_engine
import boto3

pg_url = f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'etl_user')}:{os.getenv('POSTGRES_PASSWORD', 'etl_password')}@{os.getenv('POSTGRES_HOST', 'postgres')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'oil_analytics')}"
engine = create_engine(pg_url)

with engine.connect() as conn:
    print(conn.exec_driver_sql('select current_database(), current_user').fetchone())

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT", "http://minio:9000"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin123"),
)
print([b["Name"] for b in s3.list_buckets().get("Buckets", [])])

In [ ]:
%run ../etl/export_to_minio.py